<a href="https://colab.research.google.com/github/varunsayana/jsjv-research/blob/vs/trial4_filtering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import torch
import json
import numpy as np
from typing import List, Dict, Tuple
import re
import os

drive.mount('/content/drive', force_remount=True)
DRIVE_BASE = "/content/drive/MyDrive/AlgoverseJSJV/Code/trial4_unprocessed"
OUTPUT_DIR = DRIVE_BASE

input_file = os.path.join(OUTPUT_DIR, "07_trial4_kl_rag_suppressed.json")
if os.path.exists(input_file):
    with open(input_file, "r") as f:
        data = json.load(f)
        trial4_results = []
        for cell in data.get("cells", []):
            if cell["cell_type"] == "code":
                trial4_results.append(cell["source"])

Mounted at /content/drive


In [2]:
def extract_answer_spans(text: str, answer: str, window_size: int = 50) -> List[Tuple[int, int, str]]:

    spans = []
    answer_tokens = set(answer.lower().split())

    # exact match spans
    pattern = re.compile(re.escape(answer.lower()), re.IGNORECASE)
    for match in pattern.finditer(text.lower()):
        start, end = match.span()
        window_start = max(0, start - window_size)
        window_end = min(len(text), end + window_size)
        spans.append((window_start, window_end, text[window_start:window_end]))

    # at least 70% token match spans
    tokens = text.lower().split()
    for i in range(len(tokens)):
        window_tokens = tokens[i:min(i+20, len(tokens))]
        window_set = set(window_tokens)
        overlap = len(answer_tokens & window_set)
        if overlap >= max(1, len(answer_tokens) * 0.7):
            start_char = sum(len(t) + 1 for t in tokens[:i])
            end_char = sum(len(t) + 1 for t in tokens[:min(i+20, len(tokens))])
            spans.append((start_char, end_char, text[start_char:end_char]))

    return spans

def validate_retrieved_docs(sample: Dict, retrieved_docs: List[Dict],
                           min_span_coverage: float = 0.1) -> Tuple[bool, Dict]:

    gold_answer = sample.get("gold_answer", "")

    validation_details = {
        "docs_with_spans": [],
        "span_coverage": 0.0,
        "max_overlap_score": 0.0,
        "is_valid": False
    }
    for doc_idx, doc in enumerate(retrieved_docs):
        abstract = doc.get("abstract", "")
        spans = extract_answer_spans(abstract, gold_answer)
        if spans:
            overlap_score = sum(end - start for start, end, _ in spans) / max(1, len(abstract))
            validation_details["docs_with_spans"].append({
                "doc_idx": doc_idx,
                "span_count": len(spans),
                "overlap_score": overlap_score
            })
            validation_details["max_overlap_score"] = max(
                validation_details["max_overlap_score"], overlap_score
            )
    validation_details["span_coverage"] = (
        len(validation_details["docs_with_spans"]) / max(1, len(retrieved_docs))
    )
    validation_details["is_valid"] = (
        validation_details["span_coverage"] >= min_span_coverage
    )
    return validation_details["is_valid"], validation_details

class FixedDecodingConfig:
    def __init__(self):
        self.temperature = 1.0          # No temperature scaling
        self.top_p = 1.0                # No nucleus sampling
        self.top_k = None               # No top-k filtering
        self.do_sample = False          # Greedy decoding
        self.num_beams = 1              # No beam search
        self.max_new_tokens = 50        # Fixed generation length
        self.repetition_penalty = 1.0   # No penalty
        self.length_penalty = 1.0       # No length penalty
        self.eos_token_id = None        # Let model decide

    def to_dict(self) -> Dict:
        return {
            "temperature": self.temperature,
            "top_p": self.top_p,
            "top_k": self.top_k,
            "do_sample": self.do_sample,
            "num_beams": self.num_beams,
            "max_new_tokens": self.max_new_tokens,
            "repetition_penalty": self.repetition_penalty,
            "length_penalty": self.length_penalty,
        }

In [3]:
def compute_query_relevance_score(query: str, retrieved_docs: List[Dict],
                                  similarity_scores: np.ndarray) -> float:

    if len(retrieved_docs) == 0:
        return 0.0

    doc_scores = similarity_scores[:len(retrieved_docs)]
    mean_score = float(np.mean(doc_scores))
    variance = float(np.var(doc_scores)) if len(doc_scores) > 1 else 0.0
    variance_component = 1.0 - np.exp(-variance)  # Sigmoid-like
    top_score = float(doc_scores[0])
    relevance_score = (0.5 * mean_score +
                      0.3 * variance_component +
                      0.2 * top_score)

    return min(1.0, max(0.0, relevance_score))


def filter_low_relevance_queries(trial_samples: List[Dict],
                                 relevance_threshold: float = 0.3) -> Tuple[List[Dict], Dict]:

    filtering_stats = {
        "total_samples": len(trial_samples),
        "removed_count": 0,
        "removed_indices": [],
        "relevance_scores": []
    }

    filtered_samples = []

    for sample in trial_samples:

        max_kl = sample.get("max_kl_value", 0.0)
        seq_len = sample.get("rag_seq_len", 512)

        relevance_score = float(np.exp(-max_kl / 2.0))

        filtering_stats["relevance_scores"].append(relevance_score)

        if relevance_score >= relevance_threshold:
            filtered_samples.append(sample)
        else:
            filtering_stats["removed_count"] += 1
            filtering_stats["removed_indices"].append(sample["idx"])

    filtering_stats["removal_rate"] = (
        filtering_stats["removed_count"] / filtering_stats["total_samples"]
    )

    return filtered_samples, filtering_stats

In [4]:
#rerun
def recompute_collapse_detection(samples: List[Dict],
                                collapse_threshold_multiplier: float = 0.5,
                                min_kl_layers: int = 5) -> Tuple[List[Dict], Dict]:

    recomputed_samples = []
    detection_stats = {
        "total_samples": len(samples),
        "collapse_detected": 0,
        "no_collapse_detected": 0,
        "uncertain_cases": 0,
        "threshold_values": [],
    }

    for sample in samples:
        kl_trajectory = sample.get("kl_trajectory", [])
        if len(kl_trajectory) < min_kl_layers:
            recomputed_samples.append(sample)
            detection_stats["uncertain_cases"] += 1
            continue

        valid_kl = [kl for kl in kl_trajectory[1:] if kl > 1e-6]
        if len(valid_kl) < min_kl_layers:
            recomputed_samples.append(sample)
            detection_stats["uncertain_cases"] += 1
            continue

        mean_kl = float(np.mean(valid_kl))
        final_kl = kl_trajectory[-1]
        threshold = mean_kl * collapse_threshold_multiplier

        detection_stats["threshold_values"].append(threshold)
        collapse_detected = final_kl < threshold
        sample_updated = sample.copy()
        sample_updated["collapse_detected_recomputed"] = bool(collapse_detected)
        sample_updated["collapse_threshold"] = float(threshold)
        sample_updated["collapse_final_kl"] = float(final_kl)
        sample_updated["collapse_mean_kl"] = float(mean_kl)

        recomputed_samples.append(sample_updated)
        if collapse_detected:
            detection_stats["collapse_detected"] += 1
        else:
            detection_stats["no_collapse_detected"] += 1

    detection_stats["collapse_rate_recomputed"] = (
        detection_stats["collapse_detected"] / detection_stats["total_samples"]
    )

    if detection_stats["threshold_values"]:
        detection_stats["mean_threshold"] = float(
            np.mean(detection_stats["threshold_values"])
        )
        detection_stats["std_threshold"] = float(
            np.std(detection_stats["threshold_values"])
        )

    return recomputed_samples, detection_stats

In [5]:
def apply_controlled_filtering(trial4_results: List[Dict],
                              corpus: List[Dict],
                              retriever_func,
                              span_coverage_threshold: float = 0.1,
                              relevance_threshold: float = 0.3,
                              collapse_threshold_multiplier: float = 0.5) -> Dict:

    print("\n" + "="*70)
    print("Controlled filtering pipeline")
    print("="*70)


    print("\n[1/4] Validating answer-supporting spans in retrieved documents...")
    filtered_by_spans = []
    span_validation_stats = {
        "valid_count": 0,
        "invalid_count": 0,
        "mean_span_coverage": 0.0,
        "mean_overlap_score": 0.0
    }
    span_coverages = []
    overlap_scores = []

    for result in trial4_results:
        query = result["query"]

        retrieved_docs = [{"abstract": "placeholder"}]
        is_valid, val_details = validate_retrieved_docs(
            {"gold_answer": result.get("label", "")},
            retrieved_docs,
            span_coverage_threshold
        )
        if is_valid:
            filtered_by_spans.append(result)
            span_validation_stats["valid_count"] += 1
            span_coverages.append(val_details["span_coverage"])
            overlap_scores.append(val_details["max_overlap_score"])
        else:
            span_validation_stats["invalid_count"] += 1
    if span_coverages:
        span_validation_stats["mean_span_coverage"] = float(np.mean(span_coverages))
        span_validation_stats["mean_overlap_score"] = float(np.mean(overlap_scores))

    print(f"  Valid samples: {span_validation_stats['valid_count']}")
    print(f"  Invalid samples: {span_validation_stats['invalid_count']}")
    print(f"  Retention rate: {span_validation_stats['valid_count']/len(trial4_results):.1%}")

    print("\n[2/4] Setting fixed decoding parameters...")
    decoding_config = FixedDecodingConfig()
    print(f"  Config: {decoding_config.to_dict()}")

    print("\n[3/4] Filtering low-relevance queries...")
    filtered_by_relevance, relevance_stats = filter_low_relevance_queries(
        filtered_by_spans,
        relevance_threshold
    )
    print(f"  Removed: {relevance_stats['removed_count']} samples")
    print(f"  Retention rate: {1 - relevance_stats['removal_rate']:.1%}")

    print("\n[4/4] Re-computing collapse detection on filtered set...")
    recomputed_samples, collapse_stats = recompute_collapse_detection(
        filtered_by_relevance,
        collapse_threshold_multiplier
    )
    print(f"  Collapse detected: {collapse_stats['collapse_detected']}")
    print(f"  No collapse: {collapse_stats['no_collapse_detected']}")
    print(f"  Uncertain: {collapse_stats['uncertain_cases']}")
    print(f"  Collapse rate: {collapse_stats['collapse_rate_recomputed']:.1%}")

    results = {
        "pipeline_config": {
            "span_coverage_threshold": span_coverage_threshold,
            "relevance_threshold": relevance_threshold,
            "collapse_threshold_multiplier": collapse_threshold_multiplier,
            "decoding_config": decoding_config.to_dict()
        },
        "filtering_stages": {
            "span_validation": {
                "input_count": len(trial4_results),
                "output_count": len(filtered_by_spans),
                **span_validation_stats
            },
            "relevance_filtering": {
                "input_count": len(filtered_by_spans),
                "output_count": len(filtered_by_relevance),
                **relevance_stats
            }
        },
        "collapse_detection_recomputed": {
            "input_count": len(filtered_by_relevance),
            **collapse_stats
        },
        "final_samples": recomputed_samples,
        "cumulative_retention_rate": (
            len(recomputed_samples) / len(trial4_results)
        )
    }
    # summary
    print("\n" + "="*70)
    print("Filtering Summary")
    print("="*70)
    print(f"Original samples: {len(trial4_results)}")
    print(f"After span validation: {len(filtered_by_spans)}")
    print(f"After relevance filtering: {len(filtered_by_relevance)}")
    print(f"Final samples: {len(recomputed_samples)}")
    print(f"Cumulative retention: {results['cumulative_retention_rate']:.1%}")
    print(f"Collapse rate (original): {sum(1 for r in trial4_results if r.get('collapse_detected'))}/{len(trial4_results)}")
    print(f"Collapse rate (filtered): {collapse_stats['collapse_detected']}/{len(recomputed_samples)}")
    print("="*70)

    return results

In [6]:
import os
import json
from typing import List, Dict

# results summary
corpus_DRIVE_BASE = DRIVE_BASE
DATA_PATH = os.path.join(corpus_DRIVE_BASE, "pubmedqa_filtered.json")

if os.path.exists(DATA_PATH):
    with open(DATA_PATH, "r") as f:
        corpus = json.load(f)
    print(f"Corpus loaded: {len(corpus)} samples from {DATA_PATH}")
else:
    print(f"Warning: Corpus file not found at {DATA_PATH}. 'corpus' will be an empty list.")
    corpus = []

trial4_output_file = os.path.join(OUTPUT_DIR, "trial4_full_results.json")

if os.path.exists(trial4_output_file):
    with open(trial4_output_file, "r") as f:
        full_results_data = json.load(f)
        trial4_results = full_results_data.get("samples", [])
    print(f"trial4_results loaded: {len(trial4_results)} samples from {trial4_output_file}")
else:
    print(f"Error: trial4_full_results.json not found at {trial4_output_file}.")
    print("Please ensure the previous trial (generating trial4_full_results.json) has been run successfully.")
    trial4_results = []

def hybrid_retrieve(query: str, corpus: List[Dict], top_k: int = 5) -> List[Dict]:

    print(f"Hybrid retrieval called for query: '{query}'")
    if corpus:

        return [{"abstract": d.get("abstract", ""), "id": d.get("id", i)} for i, d in enumerate(corpus[:top_k])]
    else:
        return [{"abstract": "Dummy abstract for " + query + " doc " + str(i)} for i in range(top_k)]

if not trial4_results:
    print("Skipping apply_controlled_filtering due to empty trial4_results.")
else:

    filtered_results = apply_controlled_filtering(
        trial4_results=trial4_results,
        corpus=corpus,
        retriever_func=hybrid_retrieve,
        span_coverage_threshold=0.15,      # At least 15% of docs with spans
        relevance_threshold=0.35,           # Relevance score > 0.35
        collapse_threshold_multiplier=0.5   # Original threshold
    )
    filtering_output_path = os.path.join(OUTPUT_DIR, "trial4_filtered_results.json")
    with open(filtering_output_path, "w") as f:
        json.dump(filtered_results, f, indent=2)

    print(f"Filtered results saved to {filtering_output_path}")

Corpus loaded: 759 samples from /content/drive/MyDrive/AlgoverseJSJV/Code/trial4_unprocessed/pubmedqa_filtered.json
Error: trial4_full_results.json not found at /content/drive/MyDrive/AlgoverseJSJV/Code/trial4_unprocessed/trial4_full_results.json.
Please ensure the previous trial (generating trial4_full_results.json) has been run successfully.
Skipping apply_controlled_filtering due to empty trial4_results.
